# Reporting Views & Final Validation
Create SQL views for easy dashboard consumption and validate the full pipeline output.

In [ ]:
# === Executive Dashboard View ===
# Combines production, equipment, and trend data into a single view
spark.sql('''
CREATE OR REPLACE VIEW vw_executive_dashboard AS
SELECT
    p.production_date,
    p.production_line,
    p.shift,
    SUM(p.total_units_produced) AS units_produced,
    SUM(p.total_defects) AS defects,
    ROUND(AVG(p.oee), 2) AS avg_oee,
    ROUND(AVG(p.avg_yield_pct), 2) AS avg_yield,
    ROUND(AVG(p.avg_defect_rate), 2) AS avg_defect_rate,
    MAX(p.oee_class) AS oee_classification,
    SUM(p.total_downtime_minutes) AS downtime_minutes
FROM gold_production_daily_summary p
GROUP BY p.production_date, p.production_line, p.shift
''')
print('Created vw_executive_dashboard')

In [ ]:
# === Equipment Alert View ===
# Machines that need attention based on health score
spark.sql('''
CREATE OR REPLACE VIEW vw_equipment_alerts AS
SELECT
    reading_date,
    machine_id,
    production_line,
    health_score,
    health_status,
    anomaly_rate,
    avg_temperature,
    avg_vibration,
    temp_anomaly_count,
    vibration_anomaly_count,
    temp_range,
    vibration_range
FROM gold_equipment_health_daily
WHERE health_status IN ("Critical", "At Risk")
   OR anomaly_rate > 10
ORDER BY health_score ASC
''')
print('Created vw_equipment_alerts')

In [ ]:
# === Production Trend View ===
# Weekly trends with WoW indicators for management reports
spark.sql('''
CREATE OR REPLACE VIEW vw_production_trends AS
SELECT
    production_week,
    production_line,
    weekly_units,
    weekly_defects,
    ROUND(weekly_avg_oee, 2) AS avg_oee,
    weekly_downtime,
    weekly_batches,
    ROUND(oee_wow_change, 2) AS oee_change_vs_prev_week,
    ROUND(units_wow_pct, 1) AS units_pct_change,
    trend_direction
FROM gold_weekly_trends
ORDER BY production_line, production_week
''')
print('Created vw_production_trends')

In [ ]:
# === Quality Scorecard View ===
# One row per production line with all key KPIs
spark.sql('''
CREATE OR REPLACE VIEW vw_quality_scorecard AS
SELECT
    s.production_line,
    s.total_units_all_time,
    s.total_defects_all_time,
    ROUND(s.overall_yield, 2) AS overall_yield_pct,
    ROUND(s.overall_defect_rate, 2) AS overall_defect_rate_pct,
    ROUND(s.overall_oee, 2) AS overall_oee,
    s.operating_days,
    s.products_produced,
    ROUND(s.units_per_day, 0) AS avg_daily_output,
    ROUND(s.total_downtime_all_time, 0) AS total_downtime_minutes,
    CASE
        WHEN s.overall_oee >= 85 THEN "World Class"
        WHEN s.overall_oee >= 60 THEN "Acceptable"
        WHEN s.overall_oee >= 40 THEN "Needs Improvement"
        ELSE "Critical"
    END AS oee_rating
FROM gold_line_scorecard s
''')
print('Created vw_quality_scorecard')

In [ ]:
# === Final Validation ===
print('\n' + '=' * 60)
print('MANUFACTURING QUALITY CONTROL DEMO — DEPLOYMENT SUMMARY')
print('=' * 60)

tables = {
    'Bronze': ['bronze_sensor_readings', 'bronze_production_batches', 'bronze_equipment_catalog'],
    'Silver': ['silver_sensor_readings', 'silver_production_batches', 'silver_equipment_catalog'],
    'Gold': ['gold_production_daily_summary', 'gold_equipment_health_daily',
             'gold_shift_performance', 'gold_product_quality',
             'gold_weekly_trends', 'gold_line_scorecard'],
}

for layer, tbls in tables.items():
    print(f'\n--- {layer} Layer ---')
    for t in tbls:
        try:
            cnt = spark.read.format('delta').table(t).count()
            print(f'  {t}: {cnt:,} rows')
        except Exception as e:
            print(f'  {t}: ERROR - {e}')

print('\n--- Reporting Views ---')
for v in ['vw_executive_dashboard', 'vw_equipment_alerts',
          'vw_production_trends', 'vw_quality_scorecard']:
    try:
        cnt = spark.sql(f'SELECT COUNT(*) FROM {v}').collect()[0][0]
        print(f'  {v}: {cnt:,} rows')
    except Exception as e:
        print(f'  {v}: ERROR - {e}')

print('\n=== Deployment Complete ===')